# FastSpeech2 (Small) — PyTorch Implementation

In [ ]:
import mathimport torchimport torch.nn as nnimport torch.nn.functional as Ffrom torch.utils.data import Dataset, DataLoader

In [ ]:
class Config:    vocab_size = 100    max_seq_len = 200    d_model = 128    n_heads = 2    n_layers = 2    d_ff = 256    kernel_size = 3    dropout = 0.1    n_mel = 80    variance_hidden = 128    variance_kernel = 3    variance_dropout = 0.5    pitch_min = 0.0    pitch_max = 5.0    energy_min = 0.0    energy_max = 5.0    n_bins = 256cfg = Config()device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class PositionalEncoding(nn.Module):    def __init__(self, d_model, max_len):        super().__init__()        pe = torch.zeros(max_len, d_model)        position = torch.arange(0, max_len).unsqueeze(1).float()        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))        pe[:, 0::2] = torch.sin(position * div_term)        pe[:, 1::2] = torch.cos(position * div_term)        self.register_buffer("pe", pe.unsqueeze(0))    def forward(self, x):        return x + self.pe[:, :x.size(1)]

In [ ]:
class ConvFFN(nn.Module):    def __init__(self, d_model, d_ff, kernel_size, dropout):        super().__init__()        self.conv1 = nn.Conv1d(d_model, d_ff, kernel_size, padding=kernel_size // 2)        self.conv2 = nn.Conv1d(d_ff, d_model, kernel_size, padding=kernel_size // 2)        self.dropout = nn.Dropout(dropout)        self.norm = nn.LayerNorm(d_model)    def forward(self, x):        residual = x        y = x.transpose(1, 2)        y = F.relu(self.conv1(y))        y = self.conv2(y)        y = y.transpose(1, 2)        y = self.dropout(y)        return self.norm(residual + y)

In [ ]:
class FFTBlock(nn.Module):    def __init__(self, d_model, n_heads, d_ff, kernel_size, dropout):        super().__init__()        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)        self.norm1 = nn.LayerNorm(d_model)        self.dropout1 = nn.Dropout(dropout)        self.ffn = ConvFFN(d_model, d_ff, kernel_size, dropout)    def forward(self, x, mask=None):        residual = x        attn_out, _ = self.self_attn(x, x, x, key_padding_mask=mask)        x = self.norm1(residual + self.dropout1(attn_out))        x = self.ffn(x)        return x

In [ ]:
class Encoder(nn.Module):    def __init__(self, cfg):        super().__init__()        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=0)        self.pos_enc = PositionalEncoding(cfg.d_model, cfg.max_seq_len)        self.layers = nn.ModuleList([            FFTBlock(cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.kernel_size, cfg.dropout)            for _ in range(cfg.n_layers)        ])    def forward(self, x, mask=None):        x = self.embedding(x)        x = self.pos_enc(x)        for layer in self.layers:            x = layer(x, mask)        return x

In [ ]:
class Decoder(nn.Module):    def __init__(self, cfg):        super().__init__()        self.pos_enc = PositionalEncoding(cfg.d_model, cfg.max_seq_len)        self.layers = nn.ModuleList([            FFTBlock(cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.kernel_size, cfg.dropout)            for _ in range(cfg.n_layers)        ])    def forward(self, x, mask=None):        x = self.pos_enc(x)        for layer in self.layers:            x = layer(x, mask)        return x

In [ ]:
class VariancePredictor(nn.Module):    def __init__(self, cfg):        super().__init__()        self.conv1 = nn.Conv1d(cfg.d_model, cfg.variance_hidden, cfg.variance_kernel, padding=cfg.variance_kernel // 2)        self.ln1 = nn.LayerNorm(cfg.variance_hidden)        self.conv2 = nn.Conv1d(cfg.variance_hidden, cfg.variance_hidden, cfg.variance_kernel, padding=cfg.variance_kernel // 2)        self.ln2 = nn.LayerNorm(cfg.variance_hidden)        self.dropout = nn.Dropout(cfg.variance_dropout)        self.linear = nn.Linear(cfg.variance_hidden, 1)    def forward(self, x):        y = x.transpose(1, 2)        y = F.relu(self.conv1(y)).transpose(1, 2)        y = self.dropout(self.ln1(y))        y = y.transpose(1, 2)        y = F.relu(self.conv2(y)).transpose(1, 2)        y = self.dropout(self.ln2(y))        y = self.linear(y).squeeze(-1)        return y

In [ ]:
class LengthRegulator(nn.Module):    def forward(self, x, durations, max_len=None):        output = []        for batch_idx in range(x.size(0)):            expanded = []            for t in range(x.size(1)):                repeat = max(int(durations[batch_idx, t].item()), 0)                if repeat > 0:                    expanded.append(x[batch_idx, t].unsqueeze(0).expand(repeat, -1))            if len(expanded) > 0:                expanded = torch.cat(expanded, dim=0)            else:                expanded = x.new_zeros((1, x.size(2)))            output.append(expanded)        lengths = [o.size(0) for o in output]        max_len = max_len or max(lengths)        padded = x.new_zeros((x.size(0), max_len, x.size(2)))        for i, o in enumerate(output):            padded[i, :o.size(0)] = o        return padded, torch.tensor(lengths, device=x.device)

In [ ]:
class VarianceAdaptor(nn.Module):    def __init__(self, cfg):        super().__init__()        self.duration_predictor = VariancePredictor(cfg)        self.pitch_predictor = VariancePredictor(cfg)        self.energy_predictor = VariancePredictor(cfg)        self.length_regulator = LengthRegulator()        pitch_bins = torch.linspace(cfg.pitch_min, cfg.pitch_max, cfg.n_bins - 1)        energy_bins = torch.linspace(cfg.energy_min, cfg.energy_max, cfg.n_bins - 1)        self.register_buffer("pitch_bins", pitch_bins)        self.register_buffer("energy_bins", energy_bins)        self.pitch_embedding = nn.Embedding(cfg.n_bins, cfg.d_model)        self.energy_embedding = nn.Embedding(cfg.n_bins, cfg.d_model)    def forward(self, x, duration_target=None, pitch_target=None, energy_target=None, max_len=None):        log_duration_pred = self.duration_predictor(x)        pitch_pred = self.pitch_predictor(x)        energy_pred = self.energy_predictor(x)        if pitch_target is not None:            pitch_emb = self.pitch_embedding(torch.bucketize(pitch_target, self.pitch_bins))        else:            pitch_emb = self.pitch_embedding(torch.bucketize(pitch_pred, self.pitch_bins))        x = x + pitch_emb        if energy_target is not None:            energy_emb = self.energy_embedding(torch.bucketize(energy_target, self.energy_bins))        else:            energy_emb = self.energy_embedding(torch.bucketize(energy_pred, self.energy_bins))        x = x + energy_emb        if duration_target is not None:            x, mel_lens = self.length_regulator(x, duration_target, max_len)        else:            durations = torch.clamp(torch.round(torch.exp(log_duration_pred) - 1), min=0)            x, mel_lens = self.length_regulator(x, durations, max_len)        return x, log_duration_pred, pitch_pred, energy_pred, mel_lens

In [ ]:
class FastSpeech2(nn.Module):    def __init__(self, cfg):        super().__init__()        self.encoder = Encoder(cfg)        self.variance_adaptor = VarianceAdaptor(cfg)        self.decoder = Decoder(cfg)        self.mel_linear = nn.Linear(cfg.d_model, cfg.n_mel)    def forward(self, text, duration_target=None, pitch_target=None, energy_target=None, max_mel_len=None):        x = self.encoder(text)        x, log_duration_pred, pitch_pred, energy_pred, mel_lens = self.variance_adaptor(            x, duration_target, pitch_target, energy_target, max_mel_len        )        x = self.decoder(x)        mel_out = self.mel_linear(x)        return mel_out, log_duration_pred, pitch_pred, energy_pred, mel_lens

In [ ]:
class ToyTTSDataset(Dataset):    def __init__(self, n_samples=64, cfg=cfg):        self.n_samples = n_samples        self.cfg = cfg    def __len__(self):        return self.n_samples    def __getitem__(self, idx):        text_len = torch.randint(10, 30, (1,)).item()        text = torch.randint(1, self.cfg.vocab_size, (text_len,))        durations = torch.randint(1, 5, (text_len,)).float()        pitch = torch.rand(text_len) * self.cfg.pitch_max        energy = torch.rand(text_len) * self.cfg.energy_max        mel_len = int(durations.sum().item())        mel = torch.randn(mel_len, self.cfg.n_mel)        return text, durations, pitch, energy, meldef collate_fn(batch):    texts, durations, pitches, energies, mels = zip(*batch)    text_lens = [t.size(0) for t in texts]    mel_lens = [m.size(0) for m in mels]    max_text_len = max(text_lens)    max_mel_len = max(mel_lens)    text_padded = torch.zeros(len(batch), max_text_len, dtype=torch.long)    duration_padded = torch.zeros(len(batch), max_text_len)    pitch_padded = torch.zeros(len(batch), max_text_len)    energy_padded = torch.zeros(len(batch), max_text_len)    mel_padded = torch.zeros(len(batch), max_mel_len, cfg.n_mel)    for i in range(len(batch)):        text_padded[i, :text_lens[i]] = texts[i]        duration_padded[i, :text_lens[i]] = durations[i]        pitch_padded[i, :text_lens[i]] = pitches[i]        energy_padded[i, :text_lens[i]] = energies[i]        mel_padded[i, :mel_lens[i]] = mels[i]    return text_padded, duration_padded, pitch_padded, energy_padded, mel_padded, torch.tensor(mel_lens)

In [ ]:
dataset = ToyTTSDataset(n_samples=64)loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)model = FastSpeech2(cfg).to(device)optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train_epoch(model, loader, optimizer):    model.train()    total_loss = 0.0    for text, duration, pitch, energy, mel, mel_lens in loader:        text = text.to(device)        duration = duration.to(device)        pitch = pitch.to(device)        energy = energy.to(device)        mel = mel.to(device)        max_mel_len = mel.size(1)        mel_out, log_duration_pred, pitch_pred, energy_pred, pred_mel_lens = model(            text, duration, pitch, energy, max_mel_len        )        mel_loss = F.mse_loss(mel_out, mel)        duration_loss = F.mse_loss(log_duration_pred, torch.log(duration + 1))        pitch_loss = F.mse_loss(pitch_pred, pitch)        energy_loss = F.mse_loss(energy_pred, energy)        loss = mel_loss + duration_loss + pitch_loss + energy_loss        optimizer.zero_grad()        loss.backward()        optimizer.step()        total_loss += loss.item()    return total_loss / len(loader)

In [ ]:
for epoch in range(5):    avg_loss = train_epoch(model, loader, optimizer)    print(f"epoch {epoch + 1} loss {avg_loss:.4f}")

In [ ]:
model.eval()with torch.no_grad():    text_sample = torch.randint(1, cfg.vocab_size, (1, 20)).to(device)    mel_out, log_duration_pred, pitch_pred, energy_pred, mel_lens = model(text_sample)    print(mel_out.shape, mel_lens)